# 03 — Model Comparison: local / nano / advanced / claude

**Goal**: run the same prompt through all four sky-finance model tiers and
compare output quality, latency, and cost.

The context and prompt are **fixed** so the only variable is the model.
This makes it easy to see what you actually get for the extra cost.

**Prerequisites by tier**:

| Tier | Requirement |
|------|------------|
| `local` | Ollama running: `ollama pull qwen2.5:14b-instruct` |
| `nano` | `OPENAI_API_KEY` in `.env` |
| `advanced` | `OPENAI_API_KEY` in `.env` |
| `claude` | `ANTHROPIC_API_KEY` in `.env` |

Each tier cell is **independent** — skip any tier you don't have credentials
for; the comparison table at the end shows only what ran.

In [ ]:
import os
import time
import textwrap

from sky_finance.strategies.engine import run_with_model

# Results accumulate here as each tier cell runs
results: dict[str, dict | None] = {}

def run_tier(tier: str, system: str, user: str) -> None:
    print(f'Running {tier!r} tier ...', end=' ', flush=True)
    try:
        t0 = time.perf_counter()
        text, model_id, usage = run_with_model(tier, system, user)
        elapsed = round(time.perf_counter() - t0, 1)
        results[tier] = {
            'model':         model_id,
            'text':          text,
            'elapsed':       elapsed,
            'input_tokens':  usage.input_tokens,
            'output_tokens': usage.output_tokens,
            'cost_usd':      usage.cost_usd,
        }
        cost_str = f'${usage.cost_usd:.4f}' if usage.cost_usd else 'free'
        print(f'done  {elapsed}s  in={usage.input_tokens} out={usage.output_tokens}  {cost_str}')
    except Exception as exc:
        results[tier] = None
        print(f'SKIPPED — {exc}')

def show(tier: str) -> None:
    r = results.get(tier)
    if r is None:
        print(f'  {tier}: not run')
        return
    cost_str = f'${r["cost_usd"]:.4f}' if r['cost_usd'] else 'free'
    sep = '=' * 68
    print(f'\n{sep}')
    print(f'  {tier.upper()} — {r["model"]}  ({r["elapsed"]}s  {cost_str})')
    print(sep)
    for line in r['text'].strip().splitlines()[:35]:
        print(textwrap.fill(line, 68) if len(line) > 68 else line)

print('Setup complete. Run each tier cell below, then the comparison cell.')

In [ ]:
# Fixed context and prompt — identical for all tiers.
# Keeping them constant isolates the model as the only variable.
CONTEXT = '''
### Apple Q4 Earnings Beat [positive] (sim=0.91)
Apple reported Q4 revenue of $94.9B, beating consensus estimates by 3.2 %.
iPhone revenue reached $46.2B on strong demand from China and India.
Services revenue hit a new record at $24.2B, up 16 % YoY.
EPS of $1.46 beat the $1.39 estimate; gross margin expanded 60 bps to 46.2 %.

### iPhone Supply Chain Risk [negative] (sim=0.87)
TSMC Arizona fab faces equipment installation delays, putting 15 % of iPhone
production capacity at risk through H1 next year. Apple is qualifying suppliers
in Vietnam and India as backup, but the transition could take 12-18 months.
Foxconn warned of component shortfalls affecting 3-5 % of annual volume.

### Fed Rate Decision Impact [neutral] (sim=0.71)
The Federal Reserve held rates at 5.25-5.5 % for the fourth consecutive meeting.
Technology stocks remain sensitive to rates as future earnings are discounted
at higher rates. Apple's $90B annual buyback programme benefits from lower
borrowing costs. Analysts await forward guidance on the pace of future cuts.
'''.strip()

SYSTEM = '''You are a quantitative equity analyst at a long/short hedge fund.
Be specific, cite evidence from the context, and quantify where possible.
Do not invent facts not present in the provided context.'''

USER = f'''Analyze AAPL using ONLY the evidence in the context below.
Think step by step before concluding.

Context:
{CONTEXT}

Output:
**Signal**: [Strong Bull / Bull / Neutral / Bear / Strong Bear]
**Bull case**: [2 specific evidence-backed points]
**Bear case**: [2 specific evidence-backed points]
**Outlook**: [2 sentences max]
**Confidence**: [High / Medium / Low] — [one-line reason]'''

print(f'Prompt ready: {len(USER)} chars  ({len(USER.split())} words)')

## Local tier — Ollama (free, private)

Runs entirely on your machine.  No API key, no cost, no data leaving your
network.  Trade-off: lower reasoning quality than frontier models, and
requires ~8 GB RAM (14B model with 4-bit quantisation).

In [ ]:
run_tier('local', SYSTEM, USER)
show('local')

## Nano tier — OpenAI gpt-5.4-nano

A small, fast, cheap frontier model.  Requires `OPENAI_API_KEY`.

Best for: bulk strategies that run across 20+ tickers daily, where you need
good-enough quality at low cost per ticker.

In [ ]:
run_tier('nano', SYSTEM, USER)
show('nano')

## Advanced tier — OpenAI gpt-5

Highest-quality OpenAI model.  Requires `OPENAI_API_KEY`.

Best for: deep single-stock analysis or strategies where output quality
directly drives high-stakes decisions.  Costs ~30× nano per token.

In [ ]:
run_tier('advanced', SYSTEM, USER)
show('advanced')

## Claude tier — Anthropic claude-sonnet-4-6 (with prompt caching)

Requires `ANTHROPIC_API_KEY`.

**Caching advantage**: sky-finance marks the system prompt
`cache_control: ephemeral`.  When the same strategy runs across 20 tickers,
the system prompt is cached after the *first* call — subsequent calls pay
~10 % of the system-prompt cost.  For long strategies this can make Claude
cheaper *per ticker* than nano on the first run and far cheaper on re-runs.

In [ ]:
run_tier('claude', SYSTEM, USER)
show('claude')

## Comparison table

In [ ]:
print(f'\n{"Tier":<12} {"Model":<28} {"Latency":>9} {"In tok":>7} {"Out tok":>8} {"Cost":>9}')
print('-' * 78)
for tier in ('local', 'nano', 'advanced', 'claude'):
    r = results.get(tier)
    if r is None:
        print(f'{tier:<12} {"— not run —":<28}')
        continue
    cost_str = f'${r["cost_usd"]:.5f}' if r['cost_usd'] is not None else 'free'
    print(
        f'{tier:<12} {r["model"]:<28} {r["elapsed"]:>8.1f}s'
        f' {r["input_tokens"]:>7} {r["output_tokens"]:>8} {cost_str:>9}'
    )

# Cost at scale: 20 tickers × cost_per_run
print('\nProjected cost for 20 tickers / strategy run:')
for tier in ('local', 'nano', 'advanced', 'claude'):
    r = results.get(tier)
    if r and r['cost_usd'] is not None:
        print(f'  {tier:<10} ${r["cost_usd"] * 20:.4f}')
    elif r:
        print(f'  {tier:<10} free (local)')

## When to use each tier

| Tier | Use case | Notes |
|------|---------|-------|
| `local` | Dev, experiments, air-gapped environments | Free, private, but lower reasoning quality |
| `nano` | Bulk group strategies, daily batch runs | Good quality-cost balance; ~30× cheaper than advanced |
| `advanced` | Deep single-stock analysis, high-stakes decisions | Highest quality, highest cost |
| `claude` | Long-context strategies, repeated system prompts | Prompt caching makes it cost-competitive for multi-ticker runs |

**Claude caching maths** (example with claude-sonnet-4-6):
- System prompt = 500 tokens → costs $0.0015 on the first call
- Calls 2–20 pay only the cache-read rate (~10 %) → $0.00015 each
- On a 20-ticker strategy run: system-prompt cost ≈ $0.0015 + 19 × $0.00015 ≈ **$0.0044** total
- Compared to paying full price every call: 20 × $0.0015 = **$0.030**
- **~7× saving on system-prompt tokens for long strategies**

---

**Next steps**: set a tier in the dashboard at `/strategies` and run a real
strategy against live market data.  The tier swap is a one-field edit — no
code changes needed.